# nb957 — molecule-only Uni-Mol (learned 3D) on PXR activity

**The cheapest decisive test of the 3D axis** (staged per cycle-289 decision). nb953 killed 2D-SMILES (ChemBERTa frozen worse + not flatter); nb954/nb956 showed *hand-crafted* 3D descriptors are seed-noise. The one open question: does a **learned** 209M-conformer model (Uni-Mol) extract 3D signal the hand-crafted descriptors missed? This runs molecule-only Uni-Mol (NO docking, NO pocket) — ~1 Kaggle session — and plugs its scaffold-CV OOF into the SAME nb952 degradation curve.

**Decision metric:** deep-extrapolation MAE @ sim<0.3. LGBM-combined reference = **0.5924**. If Uni-Mol beats that (flatter novel-end curve), 3D is real and the full pocket+docking pipeline is justified. If not, the 3D axis is definitively closed.

**Prereqs:** run `python scripts/kaggle_push.py --data` first to sync `unimol_*.parquet` to the dataset mount. GPU + Internet ON. Self-checkpoints per fold; re-run to resume.

**Secondary:** deploy-refit on all 4139 → 253-unblind RAE vs chemprop_aux anchor 0.6216, + 513 deploy preds saved for a possible blend test.

In [ ]:
import os, sys, time, subprocess, json, glob, shutil
import numpy as np, pandas as pd
from pathlib import Path
WORK = Path('/kaggle/working'); WORK.mkdir(exist_ok=True)
def W(m):
    with open(WORK/'trace.log','a') as f: f.write(f'[{time.strftime("%H:%M:%S")}] {m}\n')
    print(m, flush=True)
W('=== nb957 molecule-only Uni-Mol START ===')
import torch
W(f'torch={torch.__version__} cuda={torch.cuda.is_available()} '
  f'dev={torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU"}')
import numpy; W(f'numpy={numpy.__version__}')

In [ ]:
# ---- 1. Install unimol_tools (defensive; surfaces failure fast) ----
W('=== INSTALL unimol_tools ===')
t0=time.time()
r = subprocess.run([sys.executable,'-m','pip','install','-q','unimol_tools','huggingface_hub'],
                   capture_output=True, text=True, timeout=1800)
W(f'pip unimol_tools rc={r.returncode} elapsed={time.time()-t0:.0f}s')
if r.returncode!=0: W('PIP STDERR tail:\n'+r.stderr[-2500:])
# numpy guard: unimol_tools historically wants numpy<2; only downgrade if import fails
def _imp():
    import importlib
    for m in ['unimol_tools']:
        importlib.import_module(m)
try:
    _imp(); W('unimol_tools import OK on current numpy')
except Exception as e:
    W(f'import failed ({e}); trying numpy<2 + scipy pin')
    subprocess.run([sys.executable,'-m','pip','install','-q','numpy==1.26.4','scipy==1.11.4'],
                   capture_output=True, text=True, timeout=900)
    W('re-import after numpy downgrade (may need kernel restart if it still fails)')
import unimol_tools
W(f'unimol_tools {getattr(unimol_tools,"__version__","?")} ready')

In [ ]:
# ---- 2. FAIL-FAST smoke test: fit+predict on 20 molecules, validate alignment ----
W('=== SMOKE TEST (abort early if broken) ===')
from unimol_tools import MolTrain, MolPredict
smoke = pd.DataFrame({
    'SMILES': ['CCO','c1ccccc1','CC(=O)O','CCN','c1ccncc1','CCCCO','CC(C)O','c1ccc(O)cc1',
               'CCOCC','CC(=O)N','c1ccc(N)cc1','CCCl','CCBr','CC#N','CCC=O','c1ccc(F)cc1',
               'CCS','CC(C)C','c1ccc(C)cc1','CCC(=O)O'],
    'TARGET': [4.0,5.1,4.3,5.5,6.0,4.2,4.4,5.2,4.1,5.0,5.6,4.0,4.1,5.3,4.6,5.1,4.3,4.0,5.0,4.5]})
smoke.to_csv(WORK/'smoke.csv', index=False)
try:
    m = MolTrain(task='regression', data_type='molecule', epochs=2, batch_size=8,
                 metrics='mae', save_path=str(WORK/'smoke_exp'))
    m.fit(data=str(WORK/'smoke.csv'))
    p = MolPredict(load_model=str(WORK/'smoke_exp')).predict(str(WORK/'smoke.csv'))
    p = np.asarray(p).ravel()
    W(f'SMOKE OK: fit+predict ran, pred shape={p.shape} (expect 20), '
      f'finite={np.isfinite(p).sum()}')
    assert p.shape[0]==20, f'ALIGNMENT BUG: got {p.shape[0]} preds for 20 inputs'
    SMOKE_OK=True
except Exception as e:
    import traceback; W('SMOKE FAILED:\n'+traceback.format_exc()[-3000:]); SMOKE_OK=False
shutil.rmtree(WORK/'smoke_exp', ignore_errors=True)
assert SMOKE_OK, 'Smoke test failed — fix env before the full run (see trace.log).'

In [ ]:
# ---- 3. Load the bundle from the dataset mount (synced via kaggle_push.py --data) ----
def find_mount(fname):
    hits = glob.glob(f'/kaggle/input/**/{fname}', recursive=True)
    return hits[0] if hits else None
tr_path = find_mount('unimol_train.parquet')
ev_path = find_mount('unimol_eval253.parquet')
te_path = find_mount('unimol_test513.parquet')
W(f'mounts: train={tr_path} eval={ev_path} test={te_path}')
assert tr_path and ev_path and te_path, 'bundle parquet not found in /kaggle/input — run kaggle_push.py --data'
train = pd.read_parquet(tr_path); eval253 = pd.read_parquet(ev_path); test513 = pd.read_parquet(te_path)
W(f'train={train.shape} eval253={eval253.shape} test513={test513.shape}')
W(f'train cols={list(train.columns)}; folds={sorted(train.scaffold_fold.unique())}; '
  f'sim<0.3={int((train.max_sim<0.3).sum())}')
y = train['pec50'].to_numpy(float)
max_sim = train['max_sim'].to_numpy(float)
LGBM_DEEP_REF = 0.5924   # nb952 LGBM-combined deep-extrap MAE @ sim<0.3

In [ ]:
# ---- 4. 5-fold scaffold-CV fine-tune -> OOF on 4139 (RESUMABLE per fold) ----
EPOCHS=40; BS=32; LR=1e-4
def fit_predict(train_df, pred_df, exp):
    """Fit MolTrain on train_df(SMILES,TARGET), return aligned preds for pred_df."""
    tcsv=str(WORK/f'{exp}_tr.csv'); pcsv=str(WORK/f'{exp}_pred.csv')
    train_df[['SMILES','TARGET']].to_csv(tcsv, index=False)
    pred_df[['SMILES']].to_csv(pcsv, index=False)
    mdl=MolTrain(task='regression', data_type='molecule', epochs=EPOCHS, batch_size=BS,
                 learning_rate=LR, metrics='mae', save_path=str(WORK/exp))
    mdl.fit(data=tcsv)
    p=np.asarray(MolPredict(load_model=str(WORK/exp)).predict(pcsv)).ravel()
    shutil.rmtree(WORK/exp, ignore_errors=True)
    return p

oof = np.full(len(train), np.nan)
ckpt = WORK/'nb957_oof_ckpt.npy'
if ckpt.exists(): oof = np.load(ckpt); W(f'resume: {int((~np.isnan(oof)).sum())}/{len(oof)} OOF done')
for k in sorted(train['scaffold_fold'].unique()):
    va = train['scaffold_fold'].to_numpy()==k
    if not np.isnan(oof[va]).any(): W(f'fold {k}: already done, skip'); continue
    t0=time.time()
    trn = pd.DataFrame({'SMILES': train.loc[~va,'smiles'].values, 'TARGET': y[~va]})
    prd = pd.DataFrame({'SMILES': train.loc[va,'smiles'].values})
    try:
        p = fit_predict(trn, prd, f'fold{k}')
        if len(p)!=int(va.sum()):
            W(f'fold {k} ALIGN WARN: {len(p)} preds vs {int(va.sum())} rows; padding NaN')
            pp=np.full(int(va.sum()), np.nan); pp[:len(p)]=p[:int(va.sum())]; p=pp
        oof[va]=p; np.save(ckpt, oof)
        rae=lambda a,b: np.sum(np.abs(a-b))/np.sum(np.abs(a-a.mean()))
        W(f'fold {k}: n={int(va.sum())} val-RAE={rae(y[va],p):.4f} ({time.time()-t0:.0f}s)')
    except Exception as e:
        import traceback; W(f'fold {k} FAILED:\n'+traceback.format_exc()[-2000:])
W(f'OOF complete: {int((~np.isnan(oof)).sum())}/{len(oof)}')

In [ ]:
# ---- 5. Deploy: fit on ALL 4139 -> predict 513 -> 253-unblind eval + deploy preds ----
dep_ckpt = WORK/'nb957_deploy513.npy'
if dep_ckpt.exists():
    dep513=np.load(dep_ckpt); W('resume: deploy513 loaded')
else:
    t0=time.time()
    trn = pd.DataFrame({'SMILES': train['smiles'].values, 'TARGET': y})
    prd = pd.DataFrame({'SMILES': test513['smiles'].values})
    dep513 = fit_predict(trn, prd, 'deploy')
    if len(dep513)!=len(test513):
        pp=np.full(len(test513), np.nan); pp[:len(dep513)]=dep513[:len(test513)]; dep513=pp
    np.save(dep_ckpt, dep513); W(f'deploy 513 done ({time.time()-t0:.0f}s)')
pos = eval253['pos513'].to_numpy()
ev_pred = dep513[pos]; ev_y = eval253['y'].to_numpy()
rae=lambda a,b: float(np.sum(np.abs(a-b))/np.sum(np.abs(a-a.mean())))
msk=np.isfinite(ev_pred)
W(f'253-unblind deploy RAE = {rae(ev_y[msk],ev_pred[msk]):.4f}  (chemprop_aux anchor=0.6216)')

In [ ]:
# ---- 6. Degradation curve vs nb952 reference + VERDICT ----
BINS=[0.0,0.3,0.4,0.5,0.6,0.7,1.01]
fin = np.isfinite(oof)
def curve(yv,pv,sv):
    out=[]
    for lo,hi in zip(BINS[:-1],BINS[1:]):
        m=(sv>=lo)&(sv<hi)&np.isfinite(pv); n=int(m.sum())
        if n==0: out.append((f'[{lo:.1f},{hi:.1f})',0,None,None)); continue
        mae=float(np.mean(np.abs(yv[m]-pv[m])))
        rr=float(np.sum(np.abs(yv[m]-pv[m]))/np.sum(np.abs(yv[m]-yv[m].mean()))) if n>1 else None
        out.append((f'[{lo:.1f},{hi:.1f})',n,round(mae,4),round(rr,4) if rr else None))
    return out
rows=curve(y,oof,max_sim)
overall=rae(y[fin],oof[fin])
W(f'Uni-Mol molecule-only scaffold-CV OOF RAE = {overall:.4f} (LGBM-combined ref 0.5696)')
W(f"{'sim-to-train':18s} {'n':>5s} {'MAE':>8s} {'RAE':>8s}")
for b,n,mae,rr in rows:
    W(f"{b:18s} {n:5d} {(f'{mae:.4f}' if mae else '  --'):>8s} {(f'{rr:.4f}' if rr else '  --'):>8s}")
deep=next((mae for b,n,mae,rr in rows if b=='[0.0,0.3)'),None)
W('='*60)
if deep is not None:
    verdict = 'BEATS ref -> learned-3D REAL, docking justified' if deep<LGBM_DEEP_REF else 'no flatter than LGBM -> 3D axis CLOSED'
    W(f'DEEP-EXTRAP MAE @ sim<0.3: Uni-Mol={deep:.4f} vs LGBM-ref={LGBM_DEEP_REF:.4f}  <- {verdict}')
W('='*60)

In [ ]:
# ---- 7. Save outputs for local pull (kaggle_push.py --nb 957 --pull) ----
np.save(WORK/'nb957_unimol_oof_4139.npy', oof)
np.save(WORK/'nb957_unimol_deploy513.npy', dep513)
res={'overall_oof_rae': round(float(overall),4),
     'deep_extrap_mae': deep, 'lgbm_deep_ref': LGBM_DEEP_REF,
     'eval253_deploy_rae': round(rae(ev_y[msk],ev_pred[msk]),4),
     'chemprop_aux_anchor': 0.6216,
     'curve': [{'bin':b,'n':n,'mae':mae,'rae':rr} for b,n,mae,rr in rows],
     'n_oof_finite': int(fin.sum()), 'epochs': EPOCHS}
json.dump(res, open(WORK/'nb957_result.json','w'), indent=2)
pd.DataFrame({'name': test513['name'], 'smiles': test513['smiles'],
              'pEC50': dep513}).to_csv(WORK/'nb957_unimol_deploy513.csv', index=False)
W('saved: nb957_result.json, oof/deploy .npy, deploy513.csv')
print(json.dumps(res, indent=2))